# 01 — Bronze Ingest
**Pipeline:** `trades.jsonl` → Pydantic validation → Bronze Delta + Quarantine Delta  
**Layer:** Bronze (append-only, raw validated records)  
**Runs on:** Databricks cluster (Unity Catalog workspace)  
**Called by:** `flows/trade_pipeline_flow.py` → `ingest_bronze` task

## Cell 1 — Install dependencies & set up repo path

In [ ]:
# Install project dependencies on the cluster
# Run this cell once per cluster start
%pip install pydantic delta-spark --quiet

import sys
import os

# Add project root to sys.path so config.* imports resolve
# Update this path to match where you cloned the repo in your workspace

# REPO_ROOT = "/Workspace/Users/pravash.panigrahi07@gmail.com/trade-analytics-lakehouse"
REPO_ROOT = "/Volumes/workspace/default/trade-analytics/src/"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Repo root added to sys.path: {REPO_ROOT}")
print(f"Python version: {sys.version}")

## Cell 2 — Verify config imports and resolved paths

In [ ]:
sys.path.insert(0, "/Volumes/workspace/default/trade-analytics/src/")

from trade_analytics.config.enums import SPARK_ENV, SparkEnv, StorageLayer
from trade_analytics.config.settings import BRONZE_PATH, QUARANTINE_PATH
from trade_analytics.config.utils import configure_adls_auth

print(f"SPARK_ENV        : {SPARK_ENV}")
print(f"BRONZE_PATH      : {BRONZE_PATH}")
print(f"QUARANTINE_PATH  : {QUARANTINE_PATH}")

# Configure storage auth (no-op for Unity Catalog — prints confirmation)
configure_adls_auth(spark)

## Cell 3 — Verify the Unity Catalog Volume is accessible

In [ ]:
VOLUME_ROOT = "/Volumes/workspace/default/trade-analytics"

try:
    contents = dbutils.fs.ls(VOLUME_ROOT)
    print(f"Volume accessible ✓ — {len(contents)} item(s) found")
    display(dbutils.fs.ls(VOLUME_ROOT))
except Exception as e:
    print(f"Volume not accessible: {e}")
    print("Run this first: spark.sql('CREATE VOLUME IF NOT EXISTS workspace.default.`trade-analytics`')")

## Cell 4 — Upload trades.jsonl to the Volume
> **Skip this cell** if you already uploaded trades.jsonl via the Databricks CLI.  
> Only run if you want to upload directly from the notebook.

In [ ]:
# Install faker dependency required by generate_trades
%pip install faker --quiet

# Option A: Upload using dbutils (if file is already on driver node)
# dbutils.fs.cp("file:/path/to/trades.jsonl", f"{VOLUME_ROOT}/data/trades.jsonl")

# Option B: Generate fresh data directly in the notebook
import sys
sys.path.insert(0, REPO_ROOT)

from trade_analytics.data_generator.generate_trades import generate_dataset

# Write to Volume directly
OUTPUT_PATH = f"{VOLUME_ROOT}/data/trades.jsonl"
generate_dataset(n=50_000, output_path=OUTPUT_PATH)

print(f"Trades written to: {OUTPUT_PATH}")
display(dbutils.fs.ls(f"{VOLUME_ROOT}/data/"))

## Cell 5 — Define Pydantic schema

In [ ]:
from pydantic import BaseModel, validator
from typing import Literal
from datetime import datetime

class TradeEvent(BaseModel):
    trade_id:        str
    trader_id:       str
    instrument:      str
    direction:       Literal["BUY", "SELL"]
    notional:        float
    price:           float
    desk:            str
    region:          str
    counterparty:    str
    status:          Literal["EXECUTED", "CANCELLED", "PENDING"]
    trade_timestamp: datetime
    is_anomaly:      bool

    @validator("notional")
    def notional_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("notional must be positive")
        return v

    @validator("trader_id")
    def trader_id_format(cls, v):
        if not v.startswith("TDR_"):
            raise ValueError("trader_id must start with TDR_")
        return v

print("TradeEvent schema defined ✓")

## Cell 6 — Validate and split records: valid vs quarantine

In [ ]:
import json
from pydantic import ValidationError

TRADES_PATH = f"{VOLUME_ROOT}/data/trades.jsonl"

valid_records   = []
invalid_records = []

# Read JSONL from Volume using spark.read.text (serverless-compatible)
raw_lines = spark.read.text(TRADES_PATH).collect()

for row in raw_lines:
    line = row.value
    raw = json.loads(line)
    try:
        TradeEvent(**raw)
        valid_records.append(raw)
    except ValidationError as e:
        raw["_error"] = str(e)
        invalid_records.append(raw)

total     = len(valid_records) + len(invalid_records)
valid_pct = len(valid_records) / total * 100

print(f"Total records    : {total:,}")
print(f"Valid            : {len(valid_records):,}  ({valid_pct:.1f}%)")
print(f"Quarantined      : {len(invalid_records):,}  ({100 - valid_pct:.1f}%)")

## Cell 7 — Write valid records to Bronze Delta

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

if valid_records:
    df_valid = spark.createDataFrame(valid_records)

    # Add audit columns — Bronze contract: append-only + lineage
    df_valid = (
        df_valid
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source", lit("trades_jsonl"))
    )

    # Idempotent write: MERGE on trade_id prevents duplicates on re-run
    df_valid.createOrReplaceTempView("incoming_trades")

    # Check if Bronze table exists (serverless-compatible)
    try:
        spark.read.format("delta").load(BRONZE_PATH).limit(1).collect()
        bronze_exists = True
    except Exception:
        bronze_exists = False

    if bronze_exists:
        # MERGE: insert only records not already in Bronze
        spark.sql(f"""
            MERGE INTO delta.`{BRONZE_PATH}` AS target
            USING incoming_trades AS source
            ON target.trade_id = source.trade_id
            WHEN NOT MATCHED THEN INSERT *
        """)
        print("[Bronze] MERGE complete (dedup on trade_id) ✓")
    else:
        # First run — plain write
        df_valid.write.format("delta").mode("append").save(BRONZE_PATH)
        print("[Bronze] Initial write complete ✓")

    bronze_count = spark.read.format("delta").load(BRONZE_PATH).count()
    print(f"[Bronze] Total rows in table: {bronze_count:,}")
else:
    print("[Bronze] No valid records to write")

## Cell 8 — Write invalid records to Quarantine Delta

In [ ]:
if invalid_records:
    df_invalid = spark.createDataFrame(invalid_records)
    df_invalid = df_invalid.withColumn("_quarantined_at", current_timestamp())

    df_invalid.write \
        .format("delta") \
        .mode("append") \
        .save(QUARANTINE_PATH)

    quarantine_count = spark.read.format("delta").load(QUARANTINE_PATH).count()
    print(f"[Quarantine] Written {len(invalid_records):,} records")
    print(f"[Quarantine] Total rows in table: {quarantine_count:,}")
else:
    print("[Quarantine] No invalid records — data quality is clean ✓")

## Cell 9 — Register Bronze as a Unity Catalog table

In [ ]:
# Unity Catalog can query Delta tables directly from Volumes without explicit registration
# But if you want catalog tables, create them without LOCATION (managed) or verify external table support

# For now, verify the Delta tables are accessible via direct path queries
print("Bronze Delta table accessible via path query:")
display(spark.sql(f"SELECT COUNT(*) as row_count FROM delta.`{BRONZE_PATH}`"))

print("\nQuarantine Delta table check:")
try:
    display(spark.sql(f"SELECT COUNT(*) as row_count FROM delta.`{QUARANTINE_PATH}`"))
except Exception as e:
    print(f"Quarantine table not yet created (expected if no invalid records): {e}")

print("\n✓ Delta tables are accessible directly from Volume paths")
print("\nNote: To create Unity Catalog tables, use CREATE TABLE without LOCATION,")
print("or use external table registration after confirming Delta files exist.")


# spark.sql("CREATE DATABASE IF NOT EXISTS trade_analytics")

# spark.sql(f"""
#     CREATE TABLE IF NOT EXISTS trade_analytics.bronze_raw_trades
#     USING DELTA
#     LOCATION '{BRONZE_PATH}'
# """)

# spark.sql(f"""
#     CREATE TABLE IF NOT EXISTS trade_analytics.bronze_quarantine
#     USING DELTA
#     LOCATION '{QUARANTINE_PATH}'
# """)

# print("Tables registered in Unity Catalog ✓")
# display(spark.sql("SHOW TABLES IN trade_analytics"))

## Cell 10 — Bronze summary

In [ ]:
df_bronze = spark.read.format("delta").load(BRONZE_PATH)

print("\n── Bronze Summary ────────────────────────────────────")
print(f"  Total rows       : {df_bronze.count():,}")
print(f"  Schema           :")
df_bronze.printSchema()

print("\n  Status breakdown:")
display(df_bronze.groupBy("status").count().orderBy("status"))

print("\n  Sample records:")
display(df_bronze.limit(5))

print("\n── Delta Table History ───────────────────────────────")
display(spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_PATH}`"))

print("\nBronze ingest complete ✓")